# Building Blocks of the DiT

The Diffusion Transformer (DiT) is built from a small set of reusable components. In this notebook we examine each one in isolation, visualize its behavior, and understand how they fit together in the full model.

In [ ]:
import sys
sys.path.insert(0, '/fs04/scratch2/kl27/Ash/projs/Claude_projects/tutorial_GenAI/video_gen/nano-video-gen')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from einops import rearrange

from nano_video_gen.model.components import (
    sinusoidal_embedding_1d,
    precompute_freqs_cis_3d,
    rope_apply,
    RMSNorm,
    modulate,
    GateModule,
)
from nano_video_gen.model.attention import SelfAttention, CrossAttention
from nano_video_gen.visualization import (
    visualize_rope_frequencies,
    visualize_modulation_effect,
    visualize_attention_maps,
    visualize_ffn_activations,
)

torch.manual_seed(42)
print("All imports successful.")

---
## 1. Sinusoidal Embeddings

The diffusion model needs to know *which timestep* it is denoising at. A scalar timestep $t \in [0, 1000]$ is not expressive enough as a direct input, so we encode it into a high-dimensional vector using **sinusoidal positional embeddings**.

The embedding uses sine and cosine functions at exponentially spaced frequencies:

$$
\text{emb}(t)_i = 
\begin{cases}
\cos\!\left(t \cdot 10000^{-\frac{i}{d/2}}\right) & \text{if } i < d/2 \\
\sin\!\left(t \cdot 10000^{-\frac{i - d/2}{d/2}}\right) & \text{if } i \geq d/2
\end{cases}
$$

**Why sinusoidal?** Different frequencies capture patterns at different scales:
- **Low frequencies** (slow oscillation): distinguish broadly between early vs. late timesteps
- **High frequencies** (fast oscillation): distinguish between nearby timesteps

This gives the model a rich, continuous representation of "how noisy is the input right now?".

In [ ]:
# Create sinusoidal embeddings for various timesteps
dim = 64  # Nano model uses freq_dim=64

# Sample timesteps across the full range [0, 1000]
timesteps = torch.tensor([0, 50, 100, 250, 500, 750, 1000], dtype=torch.float32)
embeddings = sinusoidal_embedding_1d(dim, timesteps)

print(f"Timesteps: {timesteps.tolist()}")
print(f"Embedding shape: {list(embeddings.shape)}  (num_timesteps x dim)")
print()

# Visualize the embeddings as a heatmap
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Heatmap of all embeddings
im = axes[0].imshow(embeddings.numpy(), cmap='RdBu', aspect='auto', interpolation='nearest')
axes[0].set_xlabel('Embedding dimension')
axes[0].set_ylabel('Timestep')
axes[0].set_yticks(range(len(timesteps)))
axes[0].set_yticklabels([f't={int(t)}' for t in timesteps])
axes[0].set_title('Sinusoidal Timestep Embeddings')
plt.colorbar(im, ax=axes[0])

# Plot individual frequency channels to show the sin/cos pattern
dense_t = torch.linspace(0, 1000, 200)
dense_emb = sinusoidal_embedding_1d(dim, dense_t)

# Show a few channels: low-freq, mid-freq, high-freq
channels_to_show = [0, 8, 16, 31]  # cos channels at different frequencies
for ch in channels_to_show:
    axes[1].plot(dense_t.numpy(), dense_emb[:, ch].numpy(), label=f'dim {ch}')

axes[1].set_xlabel('Timestep')
axes[1].set_ylabel('Embedding value')
axes[1].set_title('Individual frequency channels (low to high frequency)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 2. 3D Rotary Position Embeddings (RoPE)

Video tokens live in a 3D grid: **(T, H, W)** -- temporal frame, spatial height, spatial width. The model needs to understand the relative position of each token along all three axes.

**RoPE** (Rotary Position Embeddings) encodes position by *rotating* pairs of dimensions in the query and key vectors. The key insight is that the dot product $q \cdot k$ between two rotated vectors naturally encodes their **relative** position, not their absolute position.

For 3D video, the head dimension is split into three parts:
- **Temporal** frequencies: `dim - 2*(dim//3)` dimensions
- **Height** frequencies: `dim//3` dimensions  
- **Width** frequencies: `dim//3` dimensions

Each part gets its own set of rotation angles, computed as:

$$
\theta_i = \frac{1}{10000^{2i/d}}
$$

$$
\text{freq}(\text{pos}, i) = e^{j \cdot \text{pos} \cdot \theta_i}
$$

This allows the model to learn spatial and temporal relationships independently.

In [ ]:
# Precompute 3D RoPE frequencies
head_dim = 32  # dim=128, num_heads=4 -> head_dim=32

freqs_t, freqs_h, freqs_w = precompute_freqs_cis_3d(head_dim)

print(f"Head dimension: {head_dim}")
print(f"Split: temporal={head_dim - 2*(head_dim//3)}, height={head_dim//3}, width={head_dim//3}")
print()
print(f"Temporal freqs shape: {list(freqs_t.shape)}  (max_pos x dim_t//2)")
print(f"Height freqs shape:   {list(freqs_h.shape)}  (max_pos x dim_h//2)")
print(f"Width freqs shape:    {list(freqs_w.shape)}  (max_pos x dim_w//2)")
print()

# Visualize the frequency patterns
fig = visualize_rope_frequencies(
    (freqs_t, freqs_h, freqs_w),
    max_pos=32,
)
plt.show()

---
## 3. RMSNorm

**RMSNorm** (Root Mean Square Normalization) is a simplified variant of LayerNorm used throughout the DiT.

| | LayerNorm | RMSNorm |
|---|-----------|----------|
| Mean subtraction | Yes | **No** |
| Variance normalization | Yes | Yes (via RMS) |
| Learnable affine | weight + bias | weight only |
| Speed | Baseline | ~5-10% faster |

The formula is:

$$
\text{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}} \cdot \gamma
$$

where $\gamma$ is a learnable weight vector (initialized to ones).

In Wan's architecture, RMSNorm is applied to **Q** and **K** in both self-attention and cross-attention, following the convention from LLaMA.

In [ ]:
# Create an RMSNorm instance and compare with LayerNorm
dim = 128
rms_norm = RMSNorm(dim)
layer_norm = nn.LayerNorm(dim)

# Create some random data with non-zero mean and varying scale
torch.manual_seed(42)
x = torch.randn(4, 16, dim) * 3.0 + 2.0  # mean ~2, std ~3

with torch.no_grad():
    x_rms = rms_norm(x)
    x_ln = layer_norm(x)

print(f"Input  -- mean: {x.mean():.4f}, std: {x.std():.4f}")
print(f"RMSNorm -- mean: {x_rms.mean():.4f}, std: {x_rms.std():.4f}")
print(f"LayerNorm -- mean: {x_ln.mean():.4f}, std: {x_ln.std():.4f}")
print()
print("Notice: RMSNorm does NOT center the data (mean is not forced to 0).")
print("LayerNorm centers (mean ~ 0) AND normalizes variance.")
print()

# Visualize the distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(x.flatten().numpy(), bins=80, alpha=0.7, color='gray', edgecolor='black')
axes[0].set_title(f'Input (mean={x.mean():.2f}, std={x.std():.2f})')
axes[0].axvline(x=0, color='red', linestyle='--', alpha=0.5)

axes[1].hist(x_rms.flatten().numpy(), bins=80, alpha=0.7, color='steelblue', edgecolor='black')
axes[1].set_title(f'RMSNorm (mean={x_rms.mean():.2f}, std={x_rms.std():.2f})')
axes[1].axvline(x=0, color='red', linestyle='--', alpha=0.5)

axes[2].hist(x_ln.flatten().numpy(), bins=80, alpha=0.7, color='coral', edgecolor='black')
axes[2].set_title(f'LayerNorm (mean={x_ln.mean():.2f}, std={x_ln.std():.2f})')
axes[2].axvline(x=0, color='red', linestyle='--', alpha=0.5)

for ax in axes:
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')

plt.suptitle('RMSNorm vs LayerNorm: Output Distributions', fontsize=14)
plt.tight_layout()
plt.show()

---
## 4. AdaIN Modulation

**Adaptive Instance Normalization (AdaIN)** modulation is how the timestep controls each layer's behavior. The idea is simple but powerful:

$$
\text{output} = x \cdot (1 + \text{scale}) + \text{shift}
$$

where `shift` and `scale` are **predicted from the timestep embedding**. This means:

- When `scale=0, shift=0`: output = x (identity, no modulation)
- The model learns to adjust each layer's behavior based on how noisy the input is
- Early in denoising (high noise): the model might learn large-scale adjustments
- Late in denoising (low noise): fine-grained adjustments for details

Each DiT block has **6 modulation parameters**: shift/scale/gate for self-attention, and shift/scale/gate for the FFN.

In [ ]:
# Demonstrate AdaIN modulation
dim = 128
torch.manual_seed(42)

# Input tensor (normalized by LayerNorm in practice)
x = torch.randn(1, 16, dim)

# Simulate shift and scale predicted from a timestep
# In the real model, these come from: learnable_param + time_projection(t)
shift = torch.randn(1, 1, dim) * 0.5   # moderate shift
scale = torch.randn(1, 1, dim) * 0.3   # moderate scale

# Apply modulation
x_modulated = modulate(x, shift, scale)

print(f"Input shape:      {list(x.shape)}")
print(f"Shift shape:      {list(shift.shape)}")
print(f"Scale shape:      {list(scale.shape)}")
print(f"Output shape:     {list(x_modulated.shape)}")
print()
print(f"Input  -- mean: {x.mean():.4f}, std: {x.std():.4f}")
print(f"Output -- mean: {x_modulated.mean():.4f}, std: {x_modulated.std():.4f}")
print()

# Visualize the modulation effect
fig = visualize_modulation_effect(x, x_modulated, shift, scale)
plt.show()

---
## 5. Self-Attention

**Self-attention** allows each video patch to attend to every other patch, learning both spatial and temporal relationships. In the Wan architecture:

1. Input $x$ is projected to queries $Q$, keys $K$, and values $V$
2. $Q$ and $K$ are **RMSNorm'd** (stabilizes training at scale)
3. **3D RoPE** is applied to $Q$ and $K$ (encodes position)
4. Standard scaled dot-product attention: $\text{Attn} = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V$
5. Output is projected back to model dimension

**Nano config**: dim=128, 4 heads, head_dim=32

In [ ]:
# Create a SelfAttention instance
dim = 128
num_heads = 4
head_dim = dim // num_heads  # 32

self_attn = SelfAttention(dim=dim, num_heads=num_heads)
self_attn.eval()

print(f"SelfAttention config: dim={dim}, heads={num_heads}, head_dim={head_dim}")
print(f"Parameters: {sum(p.numel() for p in self_attn.parameters()):,}")
print()

# Create dummy data: simulate a small 3D grid (e.g., 2 frames, 4x4 spatial = 32 tokens)
torch.manual_seed(42)
f, h, w = 2, 4, 4
seq_len = f * h * w  # 32 tokens
x = torch.randn(1, seq_len, dim)

# Build 3D RoPE frequencies for this grid
freqs_t, freqs_h, freqs_w = precompute_freqs_cis_3d(head_dim)
freqs = torch.cat([
    freqs_t[:f].view(f, 1, 1, -1).expand(f, h, w, -1),
    freqs_h[:h].view(1, h, 1, -1).expand(f, h, w, -1),
    freqs_w[:w].view(1, 1, w, -1).expand(f, h, w, -1)
], dim=-1).reshape(seq_len, 1, -1)

print(f"Input shape: {list(x.shape)}  (batch, seq_len={seq_len}, dim={dim})")
print(f"RoPE freqs shape: {list(freqs.shape)}")
print()

# Forward pass with a hook to capture attention weights
# We manually compute attention weights for visualization
with torch.no_grad():
    output = self_attn(x, freqs)
    print(f"Output shape: {list(output.shape)}")
    print()

    # Recompute Q, K for visualization of attention weights
    q = self_attn.norm_q(self_attn.q(x))
    k = self_attn.norm_k(self_attn.k(x))
    q = rope_apply(q, freqs, num_heads)
    k = rope_apply(k, freqs, num_heads)
    q = rearrange(q, "b s (n d) -> b n s d", n=num_heads)
    k = rearrange(k, "b s (n d) -> b n s d", n=num_heads)

    # Compute attention weights: softmax(Q K^T / sqrt(d_k))
    attn_weights = torch.matmul(q, k.transpose(-2, -1)) / (head_dim ** 0.5)
    attn_weights = F.softmax(attn_weights, dim=-1)

print(f"Attention weights shape: {list(attn_weights.shape)}  (batch, heads, seq, seq)")

# Visualize
fig = visualize_attention_maps(
    attn_weights,
    title=f"Self-Attention Maps ({seq_len} tokens = {f}f x {h}h x {w}w)",
    num_heads_to_show=4,
)
plt.show()

---
## 6. Cross-Attention

**Cross-attention** is how text prompts guide video generation. The mechanism is similar to self-attention, but:

- **Queries** come from the video features
- **Keys and Values** come from the text embeddings
- **No RoPE** (text tokens have their own positional encoding)
- $Q$ and $K$ are still RMSNorm'd

Each video token can "look at" the text tokens and pull in relevant semantic information. For example, if the prompt says "a red car driving", the cross-attention learns to route "red" information to the car regions and "driving" motion cues to the temporal dimension.

In [ ]:
# Create a CrossAttention instance
cross_attn = CrossAttention(dim=dim, num_heads=num_heads)
cross_attn.eval()

print(f"CrossAttention config: dim={dim}, heads={num_heads}")
print(f"Parameters: {sum(p.numel() for p in cross_attn.parameters()):,}")
print()

# Video features (queries): 32 video tokens
torch.manual_seed(42)
x_video = torch.randn(1, seq_len, dim)  # 32 video tokens

# Text context (keys/values): 8 text tokens (already projected to model dim)
seq_len_text = 8
context = torch.randn(1, seq_len_text, dim)  # 8 text tokens

print(f"Video input shape:  {list(x_video.shape)}  (batch, {seq_len} video tokens, dim)")
print(f"Text context shape: {list(context.shape)}  (batch, {seq_len_text} text tokens, dim)")
print()

# Forward pass
with torch.no_grad():
    output = cross_attn(x_video, context)
    print(f"Output shape: {list(output.shape)}  (same as video input)")
    print()

    # Compute cross-attention weights for visualization
    q = cross_attn.norm_q(cross_attn.q(x_video))
    k = cross_attn.norm_k(cross_attn.k(context))
    q = rearrange(q, "b s (n d) -> b n s d", n=num_heads)
    k = rearrange(k, "b s (n d) -> b n s d", n=num_heads)

    cross_attn_weights = torch.matmul(q, k.transpose(-2, -1)) / (head_dim ** 0.5)
    cross_attn_weights = F.softmax(cross_attn_weights, dim=-1)

print(f"Cross-attention weights shape: {list(cross_attn_weights.shape)}")
print(f"  -> {seq_len} video tokens attend to {seq_len_text} text tokens")

# Visualize
fig = visualize_attention_maps(
    cross_attn_weights,
    title=f"Cross-Attention Maps ({seq_len} video tokens -> {seq_len_text} text tokens)",
    num_heads_to_show=4,
)
plt.show()

---
## 7. Feed-Forward Network (FFN)

The FFN provides **non-linear transformation** capacity to each token independently. It is a simple two-layer MLP:

$$
\text{FFN}(x) = \text{Linear}_2\big(\text{GELU}(\text{Linear}_1(x))\big)
$$

The hidden dimension (`ffn_dim`) is typically 4x the model dimension:
- **Wan 14B**: dim=5120 -> ffn_dim=13824 (~2.7x)
- **Nano**: dim=128 -> ffn_dim=512 (4x)

The GELU activation (Gaussian Error Linear Unit) is a smooth approximation of ReLU that allows small negative values to pass through, which helps with gradient flow.

In [ ]:
# Create the FFN (same structure as in DiTBlock)
ffn_dim = 512
ffn = nn.Sequential(
    nn.Linear(dim, ffn_dim),
    nn.GELU(approximate='tanh'),
    nn.Linear(ffn_dim, dim)
)

print(f"FFN: Linear({dim} -> {ffn_dim}) -> GELU -> Linear({ffn_dim} -> {dim})")
print(f"Parameters: {sum(p.numel() for p in ffn.parameters()):,}")
print()

# Pass data through
torch.manual_seed(42)
x = torch.randn(1, seq_len, dim)

with torch.no_grad():
    # Capture intermediate activations
    h = ffn[0](x)          # Linear: (1, 32, 128) -> (1, 32, 512)
    h_gelu = ffn[1](h)     # GELU activation
    output = ffn[2](h_gelu) # Linear: (1, 32, 512) -> (1, 32, 128)

print(f"Input shape:        {list(x.shape)}")
print(f"After Linear 1:     {list(h.shape)}  (expanded to ffn_dim)")
print(f"After GELU:         {list(h_gelu.shape)}")
print(f"After Linear 2:     {list(output.shape)}  (back to model dim)")
print()

# Visualize FFN activations (after GELU)
fig = visualize_ffn_activations(
    h_gelu,
    title="FFN Hidden Activations (after GELU)"
)
plt.show()

---
## 8. Gate Module

The **GateModule** provides learned gating for residual connections. Instead of the standard residual pattern:

$$
\text{output} = x + f(x)
$$

Wan uses a gated version:

$$
\text{output} = x + \text{gate} \cdot f(x)
$$

where `gate` is predicted from the timestep embedding. This allows the model to control **how much** each layer contributes at each noise level:

- **High noise** (early denoising): gates might be small, letting the model make cautious updates
- **Low noise** (late denoising): gates might be large, enabling fine detail adjustments
- The gate can even be negative, allowing the model to *subtract* information

In [ ]:
# Demonstrate the gating mechanism
gate_module = GateModule()

torch.manual_seed(42)
x = torch.randn(1, seq_len, dim)       # current state
residual = torch.randn(1, seq_len, dim) # output of attention/FFN

# Test with different gate values
gate_values = [0.0, 0.5, 1.0, 2.0, -0.5]

fig, axes = plt.subplots(1, len(gate_values), figsize=(16, 3))

for i, g in enumerate(gate_values):
    gate = torch.full((1, 1, dim), g)
    output = gate_module(x, gate, residual)

    # Measure how much the output changed from x
    diff = (output - x).flatten().detach().numpy()
    axes[i].hist(diff, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
    axes[i].set_title(f'gate = {g}')
    axes[i].set_xlabel('output - x')
    axes[i].axvline(x=0, color='red', linestyle='--', alpha=0.5)
    axes[i].set_xlim(-6, 6)

axes[0].set_ylabel('Count')
plt.suptitle('GateModule: output = x + gate * residual', fontsize=14)
plt.tight_layout()
plt.show()

print("gate=0.0: No residual contribution (output = x)")
print("gate=0.5: Half the residual is added")
print("gate=1.0: Standard residual connection (output = x + residual)")
print("gate=2.0: Amplified residual (the layer contributes twice as much)")
print("gate=-0.5: Subtractive gating (the layer's effect is reversed)")

---
## Summary

We have now examined all the building blocks that make up a single DiT block:

```
DiT Block:
  1. AdaIN modulate(LayerNorm(x), shift_msa, scale_msa)
  2. Self-Attention with RoPE  -->  gated residual
  3. Cross-Attention with text  -->  additive residual
  4. AdaIN modulate(LayerNorm(x), shift_mlp, scale_mlp)
  5. FFN (Linear -> GELU -> Linear)  -->  gated residual
```

The full NanoDiT model stacks 2 of these blocks (Wan 14B uses 40), adds patchification at the input, an output head, and wraps everything with sinusoidal timestep conditioning.

Each component plays a specific role:
- **Sinusoidal Embedding**: encodes the noise level as a rich vector
- **3D RoPE**: gives the model spatial-temporal position awareness
- **RMSNorm**: stabilizes Q/K before attention
- **AdaIN Modulation**: lets the timestep control each layer's behavior
- **Self-Attention**: learns relationships between all video patches
- **Cross-Attention**: conditions generation on the text prompt
- **FFN**: provides per-token non-linear capacity
- **GateModule**: controls residual contribution per timestep